# Workflow Test: Vacuum Gripper
### Geometries, Gripper, Grasp Sampler

In [13]:
import os
from os.path import join as pjoin
import sys
import numpy as np
import open3d as o3d
import yaml
from copy import deepcopy

# Import our custom modules
# from src.grippers.vacuum_gripper import VacuumGripper, VacuumGripperConfig
from src.grippers.vacuum_gripper_v2 import VacuumGripper, VacuumGripperConfig
# from src.grasping.vacuum_sampler import VacuumGraspSampler, VacuumSamplerConfig
from src.grasping.vacuum_sampler_v2 import VacuumGraspSampler, VacuumSamplerConfig
import src.utils.geometry_utils as gu
from src.generic_geometry import GenericGeometry


### Load Configs and data

In [14]:
# franka_path = pjoin("gripper_parameter", "double_cup.yaml")
franka_path = pjoin("gripper_parameter", "single_circle_cup.yaml")
# franka_path = pjoin("gripper_parameter", "franka_vacuum.yaml")
object_path = pjoin("Test_part", "pringles", "clouds","merged_cloud.ply")

gripper = VacuumGripper(franka_path)
pcd = GenericGeometry(object_path)


✅ Geometry set: mesh (open3d)
Applying rigid transformation...
🔧 Vacuum Gripper 'Double_Schmalz_ECG' initialized.
   - Active Pads: 1
[Open3D WARNING] geometry::TriangleMesh appears to be a geometry::PointCloud (only contains vertices, but no triangles).
[Open3D] Loaded PointCloud: Test_part/pringles/clouds/merged_cloud.ply
✅ Geometry set: point_cloud (open3d)


In [15]:
gripper.visualize()
# pcd.visualize()

## Create Grasp Sampler

In [16]:
# 2. Initialize Sampler Configuration
sampler_config = VacuumSamplerConfig(
    num_samples=400,        # Cast 300 rays
    approach_distance=0.05, # 5cm approach check
    max_curvature=0.15,     # Tolerance for curvature
    min_score=0.1,           # Minimum GSS to accept
    # rotation_strategy='uniform', # 'uniform' or 'fixed'
    rotation_strategy='fixed', # 'uniform' or 'fixed'
    # rotation_step_deg=9,
    # rotation_range_deg=90,
    max_pad_gap=0.2,    # 5mm max gap
    score_aggregation_method="median",
    debug_score=True

)

# 3. Initialize Sampler
sampler = VacuumGraspSampler(gripper, sampler_config)

[VacuumSampler] Using Contact Strategy: single


### Sample Grasps

In [17]:
pcd.get_dimensions_report()

{'extents_xyz': array([0.0868, 0.0857, 0.2444]),
 'diagonal': 0.2731,
 'likely_unit': 'meters',
 'suggested_voxel_size': 0.00273,
 'details': 'Type: Open3D PointCloud. Size: 0.09 x 0.09 x 0.24. Unit: METERS.'}

In [18]:
# GenericGeometry(geometry=gripper.generate_safety_collision_mesh(
#     np.array([0,0,0]),
#     np.array([0,0,1]),
#     0.2
# )).visualize()

In [19]:
pcd_down = pcd.downsample(voxel_size=0.003)
GenericGeometry(geometry=pcd_down).visualize()

Downsampling with voxel_size=0.003...
✅ Geometry set: point_cloud (open3d)


In [20]:
sampler.config

VacuumSamplerConfig(num_samples=400, approach_distance=0.05, rotation_strategy='fixed', rotation_step_deg=12, rotation_range_deg=180, max_curvature=0.15, max_angle_deg=45.0, min_score=0.1, max_pad_gap=0.2, weight_flatness=0.4, weight_verticality=0.3, weight_torque=0.3, score_aggregation_method='median', debug_score=True)

In [21]:
# Run the sampling pipeline
# pcd_down = pcd.downsample(voxel_size=0.003)
grasps = sampler.sample_grasps(pcd_down)

print(f"\nResult: Found {len(grasps)} valid grasps.")

if len(grasps) > 0:
    best_grasp = grasps[0]
    print(f"Top Grasp Score: {best_grasp.score:.4f}")
    print(f"Top Grasp Position: {best_grasp.contact_point}")
else:
    print("No valid grasps found. Check thresholds.")

[VacuumSampler] Phase 1: Generated 400 candidates.
[VacuumSampler] Result: 8 valid grasps.

Result: Found 8 valid grasps.
Top Grasp Score: 0.8804
Top Grasp Position: [-0.00434855  0.01436228  0.24422638]


In [22]:
# for c in sampler.get_best_candidates(8):
for c in [best_grasp]:
# for c in [sampler.candidates[50]]:
    sampler.visualize_grasp(pcd_down, c)

✅ Geometry set: mesh (open3d)
Applying rigid transformation...


In [11]:
cs = filter(lambda c: c.score_details.get('failure_reason','')!='bad_angle', sampler.candidates)

In [25]:
cand_lat = sorted(sampler.candidates, key=lambda c: c.score_details.get('flatness',0), reverse=True)[0]

In [26]:
cand_lat

GraspCandidate(transform=array([[ 0.98565959, -0.11935103, -0.11929171, -0.01330714],
       [-0.11935103,  0.00667668, -0.99282967, -0.0271267 ],
       [ 0.11929171,  0.99282967, -0.00766373,  0.0590295 ],
       [ 0.        ,  0.        ,  0.        ,  1.        ]]), score=0.0, contact_point=array([-0.01330714, -0.0271267 ,  0.0590295 ]), approach_vector=array([0.11929171, 0.99282967, 0.00766373]), score_details={'flatness': np.float64(0.8454716847399467), 'verticality': 0.0, 'torque': np.float64(0.11022678639522332), 'raw_angle_deg': np.float64(89.56089630149972), 'pad_scores': {'left_cup': np.float64(0.8454716847399467)}, 'failure_reason': ['bad_angle']})

In [23]:
sampler.visualize_grasp(pcd_down, sorted(sampler.candidates, key=lambda c: c.score_details.get('flatness',0), reverse=True)[0])

✅ Geometry set: mesh (open3d)
Applying rigid transformation...


In [24]:
sampler.visualize_candidates_heatmap(pcd_down,attribute='flatness', valid_only=False, relative_scale=False)

In [14]:
sampler.visualize_candidates_heatmap(pcd_down,attribute='verticality', valid_only=False)

In [15]:
sampler.visualize_candidates_heatmap(pcd_down,attribute='torque', valid_only=False)

In [33]:
sampler.candidates[0]

GraspCandidate(transform=array([[ 0.95818531,  0.19496553, -0.20945012, -0.00742819],
       [ 0.19496553,  0.09095203,  0.97658393,  0.04974444],
       [ 0.20945012, -0.97658393,  0.04913734,  0.12795424],
       [ 0.        ,  0.        ,  0.        ,  1.        ]]), score=0.0, contact_point=array([-0.00742819,  0.04974444,  0.12795424]), approach_vector=array([-0.20945012,  0.97658393,  0.04913734]), score_details={'flatness': 0.0, 'verticality': 0.0, 'torque': 0.0, 'raw_angle_deg': 0.0, 'pad_scores': {}, 'failure_reason': 'bad_angle'})

In [14]:
sorted(sampler.candidates, key=lambda c: c.score_details.get('verticality',0), reverse=True)[0]

GraspCandidate(transform=array([[ 0.35293315,  0.93531597,  0.0249443 ,  0.02266724],
       [ 0.93531597, -0.35197155, -0.03605625,  0.00525901],
       [-0.0249443 ,  0.03605625, -0.9990384 ,  0.24682066],
       [ 0.        ,  0.        ,  0.        ,  1.        ]]), score=0.0, contact_point=array([0.02266724, 0.00525901, 0.24682066]), approach_vector=array([-0.0249443 ,  0.03605625,  0.9990384 ]), score_details={'flatness': np.float64(0.0), 'verticality': np.float64(0.9441585468554655), 'torque': np.float64(0.3253895587166594), 'raw_angle_deg': np.float64(2.5128653915040537), 'pad_scores': {'left_cup': 0.0}, 'failure_reason': ['bad_seal']})

In [16]:
sorted(sampler.candidates, key=lambda c: c.score_details.get('torque',0), reverse=True)[0]

GraspCandidate(transform=array([[-0.81829508,  0.56914597,  0.08041165, -0.00434855],
       [ 0.56914597,  0.82185117, -0.02516971,  0.01436228],
       [-0.08041165,  0.02516971, -0.9964439 ,  0.24422638],
       [ 0.        ,  0.        ,  0.        ,  1.        ]]), score=0.0, contact_point=array([-0.00434855,  0.01436228,  0.24422638]), approach_vector=array([-0.08041165,  0.02516971,  0.9964439 ]), score_details={'flatness': np.float64(0.0), 'verticality': np.float64(0.8925910013522456), 'torque': np.float64(0.9219236169670364), 'raw_angle_deg': np.float64(4.833404939148949), 'pad_scores': {'left_cup': 0.0}, 'failure_reason': ['bad_seal']})